In [14]:
import pandas as pd

In [15]:
from framework.utils import get_unseen_prefix_subset, prepare_datasets, generate_prefix_data, split_cases_temporal, split_cases, preprocess_log, get_event_log_statistics, encode_prefix_data, build_process_vocab, build_activity_vocab, count_unique_traces, filter_complete_cases


In [26]:
scenario = 'scenario_1_B_40_unique' # executed for each scenario ['scenario_1_A', 'scenario_1_B_75_unique',  'scenario_1_B_40_unique', 'scenario_1_B_20_unique', 'scenario_1_B', 'bpi2020_2processes_massive_share']
n=1000 #bpi2020_2processes_massive_share
file_name=f'dataset/RLRAM_l0.5_s00_{scenario}.csv'

file_path=file_name
p_1='p_1'
p_2='p_2'

df = pd.read_csv(file_path)
df.drop(['Unnamed: 0'], axis=1, inplace=True)
df

,method,num_processes,simulation_run,timestamp,process,l,status,case_id,activity,data,resource,end_time,cycle_time
0,RLRAM,1,0,11.036611,p_1,0.5,START,0,a_start,"{'priority': 'a_4', 'priority2': 'p1_END', 'de...",NaN,NaN,NaN
1,RLRAM,1,0,11.036611,p_1,0.5,running,0,a_1,NaN,int1,19.806278,NaN
2,RLRAM,1,0,15.213510,p_1,0.5,START,1,a_start,"{'priority': 'a_17', 'priority2': 'a_18', 'dep...",NaN,NaN,NaN
3,RLRAM,1,0,16.102863,p_1,0.5,START,2,a_start,"{'priority': 'a_2', 'priority2': 'p1_END', 'de...",NaN,NaN,NaN
4,RLRAM,1,0,18.715691,p_1,0.5,START,3,a_start,"{'priority': 'a_2', 'priority2': 'a_18', 'depa...",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
34412,RLRAM,1,0,4999.139730,p_1,0.5,gateway,1190,XOR_1,"{'priority': 'a_4', 'priority2': 'a_18', 'depa...",NaN,NaN,NaN
34413,RLRAM,1,0,4999.271475,p_1,0.5,running,1190,a_2,NaN,int2,4999.434231,NaN
34414,RLRAM,1,0,4999.434231,p_1,0.5,running,1190,a_5,NaN,int6,5000.063071,NaN
34415,RLRAM,1,0,4999.888421,p_1,0.5,gateway,1174,XOR_final_loop,"{'priority': 'a_3', 'priority2': 'a_18', 'depa...",NaN,NaN,NaN


In [27]:
print('---------------- Dataset p1-------------------')
df = filter_complete_cases(df)
df = df[['process','timestamp','case_id','activity', 'end_time','cycle_time','resource','status']]
shared_resources = len(set(df[df['process']==p_1]['resource'].unique()) & set(df[df['process']!=p_1]['resource'].unique()))
print('Number of shared resources: ',shared_resources-1)
number_of_unique_res = df[df['process']==p_1]['resource'].nunique()
print('Number of unique resources: ',number_of_unique_res)
df = df[df['process']==p_1].reset_index(drop=True)
number_of_cases = df['case_id'].nunique()
print('Number of cases: ', number_of_cases)
num_gateways = len(df[df['status']=='gateway']['activity'].unique())
print('Number of gateways: ',num_gateways)
df = df[df['status']!='gateway'].reset_index(drop=True)
unique_traces = count_unique_traces(df)
print('Number of unique traces: ', unique_traces)
unique_activities = df['activity'].nunique()
print('Number of unique activities: ', unique_activities)
print(get_event_log_statistics(df))
print('Min cycle time: ', round(df['cycle_time'].min(),2))
print('Mean cycle time: ', round(df['cycle_time'].mean(),2))
print('Max cycle time: ', round(df['cycle_time'].max(),2))
print('Std cycle time: ', round(df['cycle_time'].std(),2))
print('---------------- Dataset p2-------------------')
df_helper = pd.read_csv(file_path)
df_helper.drop(['Unnamed: 0'], axis=1, inplace=True)
df_helper = filter_complete_cases(df_helper)
df_helper = df_helper[['process','timestamp','case_id','activity', 'end_time','cycle_time','resource','status']]
df_p2 = df_helper[df_helper['process']==p_2].reset_index(drop=True)
number_of_unique_res = df_helper[df_helper['process']==p_2]['resource'].nunique()
print('Number of unique resources: ',number_of_unique_res)
number_of_cases = df_p2['case_id'].nunique()
print('Number of cases: ', number_of_cases)
num_gateways = len(df_p2[df_p2['status']=='gateway']['activity'].unique())
print('Number of gateways: ',num_gateways)
df_p2 = df_p2[df_p2['status']!='gateway'].reset_index(drop=True)
unique_traces = count_unique_traces(df_p2)
print('Number of unique traces: ', unique_traces)
unique_activities = df_p2['activity'].nunique()
print('Number of unique activities: ', unique_activities)
print(get_event_log_statistics(df_p2))
print('Min cycle time: ', round(df_p2['cycle_time'].min(),2))
print('Mean cycle time: ', round(df_p2['cycle_time'].mean(),2))
print('Max cycle time: ', round(df_p2['cycle_time'].max(),2))
print('Std cycle time: ', round(df_p2['cycle_time'].std(),2))


---------------- Dataset p1-------------------
Number of shared resources:  -1
Number of unique resources:  21
Number of cases:  1178
Number of gateways:  21
Number of unique traces:  515
Number of unique activities:  54
{'Number of Cases': 1178, 'Number of Unique Activities': 54, 'Min Case Length': 5, 'Max Case Length': 229, 'Avg Case Length': np.float64(18.73), 'Min Case Duration': np.float64(0.91), 'Max Case Duration': np.float64(261.87), 'Avg Case Duration': np.float64(49.28)}
Min cycle time:  0.91
Mean cycle time:  56.18
Max cycle time:  261.87
Std cycle time:  42.91
---------------- Dataset p2-------------------
Number of unique resources:  0
Number of cases:  0
Number of gateways:  0
Number of unique traces:  0
Number of unique activities:  0


ValueError: cannot convert float NaN to integer

In [28]:
# Step 1: Generate prefix data
train_prefix, val_prefix, test_prefix = prepare_datasets(df, temporal=True, sample_percent=1)

print("Train samples:", len(train_prefix))
print("Validation samples:", len(val_prefix))
print("Test samples:", len(test_prefix))
# Sample output format
train_prefix.head()


Train samples: 10559
Validation samples: 4085
Test samples: 6238


,process,case_id,prefix,timestamps,time_deltas,next_activity,remaining_time,prefix_time,case_end_time,timestamp_case,list_timestamps_case
0,p_1,0,[a_start],[11.036611173382113],[0.0],a_1,54.485002,11.036611,65.521613,0.000000,[0.0]
1,p_1,0,"[a_start, a_1, a_3, a_3_1, a_6, a6_notify, a6_...","[11.036611173382113, 11.036611173382113, 19.80...","[0.0, 0.0, 8.769667291593777, 0.69232529899770...",q_1,6.020541,59.501073,65.521613,48.464462,"[0.0, 0.0, 8.769667291593777, 9.46199259059148..."
2,p_1,0,"[a_start, a_1, a_3, a_3_1, a_6, a6_notify, a6_...","[11.036611173382113, 11.036611173382113, 19.80...","[0.0, 0.0, 8.769667291593777, 0.69232529899770...",a_14,14.914396,50.607217,65.521613,39.570606,"[0.0, 0.0, 8.769667291593777, 9.46199259059148..."
3,p_1,0,"[a_start, a_1, a_3, a_3_1, a_6, a6_notify, a6_...","[11.036611173382113, 11.036611173382113, 19.80...","[0.0, 0.0, 8.769667291593777, 0.69232529899770...",a6_diag,39.511770,26.009843,65.521613,14.973232,"[0.0, 0.0, 8.769667291593777, 9.46199259059148..."
4,p_1,0,"[a_start, a_1]","[11.036611173382113, 11.036611173382113]","[0.0, 0.0]",a_3,54.485002,11.036611,65.521613,0.000000,"[0.0, 0.0]"


In [29]:
# Step 2: Build the vocab from all data (or just training set if you prefer)
df1 = pd.read_csv(file_path)
df1.drop(['Unnamed: 0'], axis=1, inplace=True)
#df1 = filter_complete_cases(df1)
df1 = df1[['process','timestamp','case_id','activity', 'end_time','cycle_time','resource','status']]
df1 = df1[df1['status']!='gateway'].reset_index(drop=True)
activity_to_int, int_to_activity = build_activity_vocab(df1)
process_to_int, int_to_process = build_process_vocab(df1)


# Step 3: Encode all splits
train_prefix = encode_prefix_data(train_prefix, activity_to_int, process_to_int)
val_prefix = encode_prefix_data(val_prefix, activity_to_int, process_to_int)
test_prefix = encode_prefix_data(test_prefix, activity_to_int, process_to_int)
#test_gen_prefix = encode_prefix_data(test_gen_prefix, activity_to_int, process_to_int)

In [30]:
import pickle
pickle.dump( int_to_activity, open(f"dataset/{scenario}_int_to_activity.p", "wb" ) )
pickle.dump( activity_to_int, open(f"dataset/{scenario}_activity_to_int.p", "wb" ) )
pickle.dump( int_to_process, open(f"dataset/{scenario}_int_to_process.p", "wb" ) )
pickle.dump( process_to_int, open(f"dataset/{scenario}_process_to_int.p", "wb" ) )

In [31]:
# save
train_prefix.to_pickle(f'dataset/{scenario}_train_prefix.pkl')
val_prefix.to_pickle(f'dataset/{scenario}_val_prefix.pkl')
test_prefix.to_pickle(f'dataset/{scenario}_test_prefix.pkl')

In [32]:
# for the generic test
import pandas as pd
#n=30
def filter_distinct_test_prefix(train_df, val_df, test_df, n, random_state=None):
    # Step 1: Drop duplicates based on prefix (converted to tuple for hashability)
    test_df_unique = test_df.copy()
    test_df_unique['prefix_tuple'] = test_df_unique['prefix'].apply(tuple)
    test_df_unique = test_df_unique.drop_duplicates(subset='prefix_tuple')

    # Step 2: Sample n unique prefixes
    if len(test_df_unique) < n:
        raise ValueError(f"Not enough unique prefixes in test_df to sample {n} rows.")
    
    test_sample = test_df_unique.sample(n=n, random_state=random_state).drop(columns='prefix_tuple').reset_index(drop=True)

    # Step 3: Create a set of sampled prefixes
    sampled_prefixes = set(tuple(prefix) for prefix in test_sample['prefix'])

    # Step 4: Define a helper to remove matching prefixes
    def remove_matching_prefixes(df, prefixes_to_remove):
        return df[~df['prefix'].apply(lambda x: tuple(x) in prefixes_to_remove)].reset_index(drop=True)

    # Step 5: Filter train and val
    train_df_filtered = remove_matching_prefixes(train_df, sampled_prefixes)
    val_df_filtered = remove_matching_prefixes(val_df, sampled_prefixes)

    return train_df_filtered, val_df_filtered, test_sample

train_prefix_gen, val_prefix_gen, test_prefix_gen = filter_distinct_test_prefix(train_prefix, val_prefix, test_prefix, n=n, random_state=42)


In [33]:
print("Train GEN samples:", len(train_prefix_gen))
print("Validation GEN samples:", len(val_prefix_gen))
print("Test GEN samples:", len(test_prefix_gen))

Train GEN samples: 7335
Validation GEN samples: 2795
Test GEN samples: 1000


In [34]:
# save
train_prefix_gen.to_pickle(f'dataset/{scenario}_train_prefix_gen.pkl')
val_prefix_gen.to_pickle(f'dataset/{scenario}_val_prefix_gen.pkl')
test_prefix_gen.to_pickle(f'dataset/{scenario}_test_prefix_gen.pkl')